# Dataanalys av Arsenal FC-spelare från 2000-talet

Det här är min Kunskapskontroll 1 i kursen Artificiell Intelligens - programmering Python.

Jag analyserar ett eget CSV-dataset med Arsenal FC-spelare som varit relevanta för klubben från 2000-talet och framåt. Datasetet är sammanställt från offentlig spelarstatistik, främst Wikipedia-listan över Arsenal-spelare: https://en.wikipedia.org/wiki/List_of_Arsenal_F.C._players

Varje rad representerar en spelare. Kolumnerna beskriver bland annat spelarens nationalitet, position, år i klubben, antal starter, inhopp, totala matcher och mål.

Målet är att undersöka frågor som:

- Vilka positioner är vanligast i datasetet?
- Vilka spelare har gjort flest mål?
- Hur hänger antal matcher och mål ihop?
- Vilka nationaliteter är vanligast?

Eftersom datasetet är manuellt sammanställt ska resultaten tolkas försiktigt. Det är inte en komplett historisk databas över alla Arsenal-spelare, utan ett urval som passar för grundläggande dataanalys.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Inläsning av datasetet

Först läser jag in CSV-filen med `pandas`. Jag sparar tabellen i variabeln `df`, som är ett vanligt namn för en DataFrame.

In [ ]:
df = pd.read_csv("data/arsenal_squad.csv")

df.head()

## 2. Mekanisk inspektion

Innan jag börjar analysera datan gör jag en mekanisk inspektion. Jag kontrollerar storlek, datatyper, enkel statistik och saknade värden.

In [ ]:
# Antal rader och kolumner
df.shape

In [ ]:
# Kolumner, datatyper och antal ifyllda värden
df.info()

In [ ]:
# Statistik för numeriska kolumner
df.describe()

In [ ]:
# Saknade värden per kolumn
df.isna().sum()

## 3. Datatvätt och förberedelse

Datasetet är redan ganska rent, men jag kontrollerar ändå dubbletter och saknade värden. Jag skapar också nya kolumner som gör analysen enklare.

In [ ]:
# Jag arbetar vidare med en kopia så att originalet finns kvar
df_clean = df.copy()

df_clean.duplicated().sum()

In [ ]:
# Ta bort eventuella dubbletter och rader med saknade värden
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna()

df_clean.shape

In [ ]:
# Skapa nya kolumner med vektoriserade beräkningar
df_clean["years_at_club"] = df_clean["club_end"] - df_clean["club_start"]
df_clean["goals_per_appearance"] = np.where(
    df_clean["total_appearances"] > 0,
    df_clean["goals"] / df_clean["total_appearances"],
    0
)

df_clean[["name", "total_appearances", "goals", "years_at_club", "goals_per_appearance"]].head()

Här använder jag `numpy` med `np.where()`. Det är en vektoriserad beräkning: i stället för att loopa rad för rad räknas mål per match ut för hela kolumnen samtidigt.

In [ ]:
# Kontroll efter datatvätt
df_clean.isna().sum()

## 4. Filtrering och gruppering med pandas

Här visar jag två vanliga pandas-operationer: boolean masking och groupby. Det hjälper mig att sammanfatta datan innan jag visualiserar den.

In [ ]:
# Boolean masking: filtrera fram anfallare med minst 50 mål
high_scoring_forwards = df_clean[
    (df_clean["position_group"] == "Forward") &
    (df_clean["goals"] >= 50)
]

high_scoring_forwards[["name", "goals", "total_appearances"]].sort_values("goals", ascending=False)

In [ ]:
# Groupby: sammanfatta spelare per positionsgrupp
position_summary = df_clean.groupby("position_group").agg(
    players=("name", "count"),
    total_goals=("goals", "sum"),
    average_appearances=("total_appearances", "mean"),
    average_goals_per_appearance=("goals_per_appearance", "mean")
).reset_index()

position_summary

## 5. Utforskning genom visualisering

Nu skapar jag fyra visualiseringar. Varje diagram utgår från en fråga, eftersom diagramtypen ska väljas utifrån vad betraktaren ska kunna jämföra.

### Fråga 1: Vilka positionsgrupper är vanligast?

Här använder jag ett stapeldiagram eftersom jag jämför antal spelare mellan kategorier.

In [ ]:
position_counts = df_clean["position_group"].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    x=position_counts.index,
    y=position_counts.values,
    ax=ax
)

ax.set_title("Midfielders and defenders dominates")
ax.set_xlabel("position")
ax.set_ylabel("Number of players")
ax.set_ylim(0, position_counts.max() + 5)

plt.show()

Diagrammet visar att vissa positionsgrupper är vanligare än andra i datasetet. Det är rimligt eftersom en trupp ofta har fler utespelare än målvakter.

### Fråga 2: Vilka spelare har gjort flest mål?

Här använder jag ett horisontellt stapeldiagram. Det passar bra eftersom spelarnamnen är långa och jag vill jämföra mål mellan spelare.

In [ ]:
top_goals = df_clean.sort_values("goals", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=top_goals,
    y="name",
    x="goals",
    ax=ax
)

ax.set_title("Thierry Henry with the most goals")
ax.set_xlabel("Goals for Arsenal")
ax.set_ylabel("Player")
ax.set_xlim(0, top_goals["goals"].max() + 25)

henry = top_goals.iloc[0]
ax.annotate(
    "228 mål",
    xy=(henry["goals"], henry["name"]),
    xytext=(henry["goals"] + 5, henry["name"]),
    va="center"
)

plt.show()

Det blir tydligt att Thierry Henry sticker ut kraftigt i datasetet. Diagrammet visar mer än tabellen eftersom skillnaden mellan honom och övriga målskyttar syns direkt.

### Fråga 3: Betyder fler matcher alltid fler mål?

Här använder jag ett punktdiagram eftersom både matcher och mål är numeriska variabler. Färgen visar position för att se om position påverkar mönstret.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=df_clean,
    x="total_appearances",
    y="goals",
    hue="position_group",
    ax=ax
)

ax.set_title("More games doesnt always result in more goals")
ax.set_xlabel("Total games")
ax.set_ylabel("Goals for Arsenal")
ax.set_xlim(0, df_clean["total_appearances"].max() + 30)
ax.set_ylim(0, df_clean["goals"].max() + 25)

ax.annotate(
    "Henry",
    xy=(377, 228),
    xytext=(420, 215),
    arrowprops={"arrowstyle": "->"}
)

plt.show()

Punktdiagrammet visar att sambandet mellan matcher och mål beror mycket på position. Målvakter och försvarare kan ha många matcher utan att göra många mål, medan anfallare oftare ligger högre på målaxeln.

### Fråga 4: Vilka nationaliteter är vanligast?

Här använder jag ett horisontellt stapeldiagram eftersom jag jämför antal spelare mellan länder och vill att etiketterna ska vara lätta att läsa.

In [ ]:
top_nationalities = df_clean["nationality"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    x=top_nationalities.values,
    y=top_nationalities.index,
    ax=ax
)

ax.set_title("England and France are the most common")
ax.set_xlabel("Number of players")
ax.set_ylabel("Nationality")
ax.set_xlim(0, top_nationalities.max() + 3)

plt.show()

Diagrammet visar att England är vanligt, vilket är väntat eftersom Arsenal är en engelsk klubb. Samtidigt syns också hur internationellt laget varit under 2000-talet, särskilt med många franska spelare.

## 6. Avslutning

Analysen visar att datasetet innehåller många mittfältare och försvarare, att Thierry Henry sticker ut som målskytt och att position påverkar hur man ska tolka målstatistik. En försvarare kan vara en väldigt viktig spelare även om målkolumnen är låg.

Det jag inte kan veta från datan är till exempel spelarnas kvalitet, skador, speltid per match, assist, titlar eller exakt roll i laget. Datasetet är också ett urval, inte en fullständig databas över alla Arsenal-spelare. Därför kan jag se mönster i urvalet, men jag ska vara försiktig med att dra för stora slutsatser om hela klubbens historia.